# Netflix Movies & Shows Explorer
Colab-ready: Kaggle fetch with same-schema fallback so Run All always works.

In [ ]:
%pip install -q pandas matplotlib kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATASET = "shivamb/netflix-shows"
try:
    import kagglehub
    kpath = kagglehub.dataset_download(DATASET)
    csvs = [os.path.join(kpath, f) for f in os.listdir(kpath) if f.endswith(".csv")]
    df = pd.read_csv(csvs[0]); source = "kagglehub"
    print("Loaded Kaggle:", csvs[0], df.shape)
except Exception as e:
    print("kagglehub skipped:", type(e).__name__)
    rng = np.random.default_rng(7); n = 6000
    types = rng.choice(["Movie", "TV Show"], size=n, p=[0.7, 0.3])
    df = pd.DataFrame({
        "show_id": [f"s{i}" for i in range(1, n+1)], "type": types,
        "title": [f"Title {i}" for i in range(1, n+1)],
        "country": rng.choice(["United States","India","United Kingdom","Canada","France","Japan","Unknown"], size=n),
        "date_added": pd.to_datetime(rng.choice(pd.date_range("2015-01-01","2021-12-31"), n)),
        "release_year": rng.integers(1990, 2022, n),
        "rating": rng.choice(["TV-MA","TV-14","TV-PG","R","PG-13","PG"], size=n),
        "duration": [f"{m} min" if t=="Movie" else f"{s} Seasons" for t,m,s in zip(types, rng.integers(70,180,n), rng.integers(1,6,n))],
        "listed_in": rng.choice(["Dramas","Comedies","Documentaries","Action","Horror"], size=n),
        "description": ["A story."]*n})
    source = "synthetic"
    print("Using synthetic fallback", df.shape)
df.head(3)

## Cleaning

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df = df.dropna(subset=["title","type"])
df["country"] = df["country"].replace(["nan","None",""], float("nan")).fillna("Unknown")
df = df.drop_duplicates("show_id")
print(len(df), "rows ready")

## Content strategy EDA

In [ ]:
df["type"].value_counts().plot(kind="bar"); plt.title("Movies vs TV Shows"); plt.xlabel("Type"); plt.ylabel("Titles"); plt.show()
df["date_added"].dt.year.value_counts().sort_index().plot(marker="o"); plt.title("Titles added per year"); plt.xlabel("Year"); plt.ylabel("Titles"); plt.show()
df.loc[df["country"]!="Unknown","country"].value_counts().head(10).plot(kind="barh"); plt.title("Top countries"); plt.xlabel("Titles"); plt.show()
df["listed_in"].str.split(", ").explode().value_counts().head(10).plot(kind="barh"); plt.title("Top genres"); plt.xlabel("Titles"); plt.show()
df["rating"].value_counts().head(8).plot(kind="bar"); plt.title("Maturity ratings"); plt.xlabel("Rating"); plt.ylabel("Titles"); plt.show()
_mv = df[df["type"]=="Movie"]["duration"].str.extract(r"(\d+)").astype(float)
_mv.hist(bins=30); plt.title("Movie lengths"); plt.xlabel("Minutes"); plt.show()

## Takeaways
- ~70/30 movie-to-show split; US + India lead supply; comedies/docs top genres.
- Family-heavy ratings = breadth strategy.
- Rerun on real Kaggle data before publishing numbers.